# DS4200 – V3 (Updated)
**Median Pollutant Concentration by U.S. Region (2025)**

### Changes based on peer feedback:
- **Removed redundant color encoding** — bars were colored by the same value on the x-axis, which added noise without new info. Now a clean single color.
- **Added `n=` station count** in each region label so viewers know how many monitoring stations back each bar.
- **Added pollutant dropdown** — switch between CO (ppm) and NO₂ (ppb) without rerunning. Both CSVs share identical EPA column names.
- **`width="container"`** — chart fills page width automatically, no horizontal scrolling.
- Subtitle now explains what the median represents and what `n` means.

In [5]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [6]:
region_map = {
    "Maine": "Northeast", "New Hampshire": "Northeast", "Vermont": "Northeast",
    "Massachusetts": "Northeast", "Rhode Island": "Northeast", "Connecticut": "Northeast",
    "New York": "Northeast", "New Jersey": "Northeast", "Pennsylvania": "Northeast",
    "Maryland": "Mid-Atlantic", "Delaware": "Mid-Atlantic",
    "District Of Columbia": "Mid-Atlantic", "Virginia": "Mid-Atlantic",
    "West Virginia": "Mid-Atlantic",
    "North Carolina": "Southeast", "South Carolina": "Southeast",
    "Georgia": "Southeast", "Florida": "Southeast", "Alabama": "Southeast",
    "Mississippi": "Southeast", "Tennessee": "Southeast", "Kentucky": "Southeast",
    "Texas": "South Central", "Oklahoma": "South Central",
    "Arkansas": "South Central", "Louisiana": "South Central",
    "Ohio": "Midwest", "Indiana": "Midwest", "Illinois": "Midwest",
    "Michigan": "Midwest", "Wisconsin": "Midwest", "Minnesota": "Midwest",
    "Iowa": "Midwest", "Missouri": "Midwest", "North Dakota": "Midwest",
    "South Dakota": "Midwest", "Nebraska": "Midwest", "Kansas": "Midwest",
    "Montana": "Mountain West", "Idaho": "Mountain West", "Wyoming": "Mountain West",
    "Colorado": "Mountain West", "New Mexico": "Mountain West",
    "Arizona": "Mountain West", "Utah": "Mountain West", "Nevada": "Mountain West",
    "Washington": "Pacific Northwest", "Oregon": "Pacific Northwest",
    "California": "California",
    "Alaska": "Alaska / Hawaii", "Hawaii": "Alaska / Hawaii",
    "Puerto Rico": "Territories",
}

In [7]:
# Both EPA CSVs share identical column names — same loader works for both.
# CO  (daily_42101_2025.csv): Arithmetic Mean in Parts per million
# NO2 (daily_42602_2025.csv): Arithmetic Mean in Parts per billion

def load_and_aggregate(csv_path, pollutant_label, unit_label):
    df = pd.read_csv(csv_path, low_memory=False)
    df.columns = df.columns.str.strip()
    df["Region"] = df["State Name"].map(region_map)

    station_counts = (
        df.dropna(subset=["Region"])
        .groupby("Region")["Site Num"]
        .nunique()
        .reset_index()
        .rename(columns={"Site Num": "n_stations"})
    )

    agg = (
        df.dropna(subset=["Region", "Arithmetic Mean"])
        .groupby("Region", as_index=False)["Arithmetic Mean"]
        .median()
        .rename(columns={"Arithmetic Mean": "Median Value"})
        .merge(station_counts, on="Region")
        .sort_values("Median Value", ascending=False)
    )

    agg["Pollutant"]     = pollutant_label
    agg["Unit"]          = unit_label
    agg["region_label"]  = agg.apply(
        lambda r: f"{r['Region']}  (n={r['n_stations']})", axis=1
    )
    return agg

co_agg  = load_and_aggregate("daily_42101_2025.csv", "CO (ppm)",   "ppm")
no2_agg = load_and_aggregate("daily_42602_2025.csv", "NO\u2082 (ppb)", "ppb")

combined = pd.concat([co_agg, no2_agg], ignore_index=True)
combined.head()

,Region,Median Value,n_stations,Pollutant,Unit,region_label
0,Southeast,0.254167,24,CO (ppm),ppm,Southeast (n=24)
1,South Central,0.245293,17,CO (ppm),ppm,South Central (n=17)
2,California,0.225000,27,CO (ppm),ppm,California (n=27)
3,Midwest,0.222136,32,CO (ppm),ppm,Midwest (n=32)
4,Mid-Atlantic,0.200000,14,CO (ppm),ppm,Mid-Atlantic (n=14)


In [8]:
# Dropdown to switch between pollutants
pollutant_select = alt.binding_select(
    options=["CO (ppm)", "NO\u2082 (ppb)"],
    name="Pollutant: "
)
pollutant_param = alt.param(
    name="selected_pollutant",
    value="CO (ppm)",
    bind=pollutant_select
)

base = alt.Chart(combined).transform_filter(
'datum.Pollutant == selected_pollutant'
)

bars = base.mark_bar(cornerRadiusEnd=4, color="#4A7FA5").encode(
    y=alt.Y(
        "region_label:N",
        sort=alt.EncodingSortField(field="Median Value", order="descending"),
        title=None,
        axis=alt.Axis(labelFontSize=12)
    ),
    x=alt.X(
        "Median Value:Q",
        title="Median Concentration",
        scale=alt.Scale(zero=True)
    ),
    tooltip=[
        alt.Tooltip("Region:N",       title="Region"),
        alt.Tooltip("Median Value:Q", title="Median Value",       format=".4f"),
        alt.Tooltip("Unit:N",         title="Unit"),
        alt.Tooltip("n_stations:Q",   title="Monitoring stations")
    ]
)

value_labels = base.mark_text(
    align="left", dx=5, fontSize=11, color="#444"
).encode(
    y=alt.Y(
        "region_label:N",
        sort=alt.EncodingSortField(field="Median Value", order="descending")
    ),
    x=alt.X("Median Value:Q"),
    text=alt.Text("Median Value:Q", format=".3f")
)

v3 = alt.layer(bars, value_labels).add_params(
    pollutant_param
).properties(
    title=alt.TitleParams(
        text="Median Daily Pollutant Concentration by U.S. Region (2025)",
        subtitle=[
            "Each bar shows the median across all daily readings in that region.",
            "n = number of unique monitoring stations contributing to the estimate."
        ],
        fontSize=16,
        subtitleFontSize=12,
        subtitleColor="#666",
        anchor="start"
    ),
    width="container",
    height=380
).configure_view(
    stroke=None
).configure_axis(
    grid=True,
    gridColor="#eeeeee"
)

v3

alt.LayerChart(...)

In [9]:
v3.save("median_pollutant_by_region.html")